# Notebook 01 — Baseline Results Analysis

Loads all 6 baseline model results (3 seeds each), generates:
- Cross-model accuracy comparison table
- Training loss & validation accuracy curves
- Per-class F1 heatmap (41 subreddits)
- F1 macro/weighted comparison bar chart

In [ ]:
import sys, os
# Add project root to path — run from within reddit_gnn/
sys.path.insert(0, os.path.abspath('..'))

import json, csv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import pandas as pd
from IPython.display import display

from reddit_gnn.config import RESULTS_ROOT, SEEDS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid')

MODEL_NAMES   = ['graphsage', 'graphsaint', 'sgc', 'gat', 'gatv2', 'cluster_gcn']
DISPLAY_NAMES = ['GraphSAGE', 'GraphSAINT', 'SGC', 'GAT', 'GATv2', 'ClusterGCN']
SAVE_DIR = os.path.join(RESULTS_ROOT, 'figures', '01_baselines')
os.makedirs(SAVE_DIR, exist_ok=True)
print('Results root:', RESULTS_ROOT)

## 1. Load All Baseline Metrics

In [ ]:
def load_metrics(model, ablation='baseline', variant='default'):
    results = []
    for seed in SEEDS:
        path = os.path.join(RESULTS_ROOT, model, ablation, variant, f'seed{seed}', 'metrics.json')
        if os.path.exists(path):
            with open(path) as f:
                results.append(json.load(f))
        else:
            print(f'  Missing: {path}')
    return results

def load_history(model, seed=0, ablation='baseline', variant='default'):
    path = os.path.join(RESULTS_ROOT, model, ablation, variant, f'seed{seed}', 'history.csv')
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return list(csv.DictReader(f))

all_results = {m: load_metrics(m) for m in MODEL_NAMES}
print('Loaded results for:', [m for m, r in all_results.items() if r])

## 2. Summary Table — Accuracy & F1 (mean ± std across 3 seeds)

In [ ]:
rows = []
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    metrics_list = all_results.get(model, [])
    if not metrics_list:
        rows.append({'Model': display, 'Acc Mean': 'N/A', 'Acc Std': 'N/A',
                     'F1 Macro': 'N/A', 'F1 Weighted': 'N/A', 'Seeds': 0})
        continue
    accs = [m['test_acc'] for m in metrics_list]
    f1m  = [m.get('f1_macro', 0) for m in metrics_list]
    f1w  = [m.get('f1_weighted', 0) for m in metrics_list]
    rows.append({
        'Model': display,
        'Acc Mean': f'{np.mean(accs):.4f}',
        'Acc Std':  f'{np.std(accs):.4f}',
        'F1 Macro': f'{np.mean(f1m):.4f}',
        'F1 Weighted': f'{np.mean(f1w):.4f}',
        'Seeds': len(metrics_list)
    })

df_summary = pd.DataFrame(rows)
display(df_summary.style.set_caption('Baseline Results — All 6 Models'))

## 3. Accuracy Comparison Bar Chart

In [ ]:
means, stds, names = [], [], []
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    ml = all_results.get(model, [])
    if ml:
        accs = [m['test_acc'] for m in ml]
        means.append(np.mean(accs)); stds.append(np.std(accs)); names.append(display)

if means:
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = sns.color_palette('husl', len(names))
    bars = ax.bar(names, means, yerr=stds, capsize=6, color=colors, edgecolor='black', lw=0.7)
    ax.axhline(0.93, color='red', ls='--', lw=1.2, alpha=0.7, label='93% gate')
    ax.set_ylabel('Test Accuracy', fontsize=13)
    ax.set_title('Baseline Test Accuracy — All 6 Models (mean ± std, 3 seeds)', fontsize=14)
    ax.set_ylim(0.88, 1.005)
    ax.legend()
    for bar, m in zip(bars, means):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
                f'{m:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,'accuracy_comparison.png'), dpi=150)
    plt.show()

## 4. Training Curves (Loss & Val Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    hist = load_history(model)
    if hist is None: continue
    epochs    = [int(h['epoch']) for h in hist]
    train_loss = [float(h['train_loss']) for h in hist]
    val_acc    = [float(h['val_acc']) for h in hist]
    axes[0].plot(epochs, train_loss, label=display, lw=1.8)
    axes[1].plot(epochs, val_acc,    label=display, lw=1.8)

for ax, title, ylabel in zip(axes,
    ['Training Loss', 'Validation Accuracy'],
    ['Cross-Entropy Loss', 'Accuracy']):
    ax.set_xlabel('Epoch', fontsize=12); ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'training_curves.png'), dpi=150)
plt.show()

## 5. Per-Class F1 Heatmap (41 Subreddits)

In [ ]:
f1_matrix, valid_names = [], []
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    ml = all_results.get(model, [])
    if ml and 'f1_per_class' in ml[0]:
        f1_matrix.append(ml[0]['f1_per_class']); valid_names.append(display)

if f1_matrix:
    fig, ax = plt.subplots(figsize=(22, max(4, len(valid_names)*1.2)))
    im = ax.imshow(f1_matrix, aspect='auto', cmap='RdYlGn', vmin=0.7, vmax=1.0)
    ax.set_yticks(range(len(valid_names))); ax.set_yticklabels(valid_names, fontsize=11)
    ax.set_xlabel('Class Index (Subreddit)', fontsize=12)
    ax.set_title('Per-Class F1 Score — 41 Subreddit Classes', fontsize=14)
    plt.colorbar(im, ax=ax, label='F1 Score')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,'per_class_f1.png'), dpi=150)
    plt.show()
else:
    print('No per-class F1 data available yet. Run baselines first.')

## 6. F1 Macro vs F1 Weighted Comparison

In [ ]:
plot_data = []
for model, display in zip(MODEL_NAMES, DISPLAY_NAMES):
    ml = all_results.get(model, [])
    if ml:
        plot_data.append({
            'Model': display,
            'F1 Macro':    np.mean([m.get('f1_macro',0) for m in ml]),
            'F1 Weighted': np.mean([m.get('f1_weighted',0) for m in ml]),
        })

if plot_data:
    df_f1 = pd.DataFrame(plot_data).set_index('Model')
    df_f1.plot(kind='bar', figsize=(12,6), color=['steelblue','darkorange'], edgecolor='black', lw=0.5)
    plt.title('F1 Macro vs F1 Weighted — All 6 Models', fontsize=14)
    plt.ylabel('F1 Score', fontsize=12)
    plt.xticks(rotation=30, ha='right')
    plt.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR,'f1_comparison.png'), dpi=150)
    plt.show()